In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

In [2]:
import os

# 현재 노트북 기준 data 폴더 경로
base_dir = os.getcwd()  # C:/dev/project/SKN27-2nd-4TEAM
data_path = os.path.join(base_dir, 'data', 'dataset.xlsx')

# 데이터 로드
df = pd.read_excel(data_path, sheet_name='E Comm')
print(f"데이터 로드 완료: {df.shape}")
df.head()

데이터 로드 완료: (5630, 20)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.0,Mobile Phone,3,6.0,Debit Card,Female,3.0,3,Laptop & Accessory,2,Single,9,1,11.0,1.0,1.0,5.0,159.93
1,50002,1,NaN,Phone,1,8.0,UPI,Male,3.0,4,Mobile,3,Single,7,1,15.0,0.0,1.0,0.0,120.90
2,50003,1,NaN,Phone,1,30.0,Debit Card,Male,2.0,4,Mobile,3,Single,6,1,14.0,0.0,1.0,3.0,120.28
3,50004,1,0.0,Phone,3,15.0,Debit Card,Male,2.0,4,Laptop & Accessory,5,Single,8,0,23.0,0.0,1.0,3.0,134.07
4,50005,1,0.0,Phone,1,12.0,CC,Male,NaN,3,Mobile,5,Single,3,0,11.0,1.0,1.0,3.0,129.60


In [3]:
def reset_seeds(seed=42): #? 이 숫자가 뭘 의미함?
  random.seed(seed)
  os.environ['PYTHONHASHSEED'] = str(seed)    # 파이썬 환경변수 시드 고정
  np.random.seed(seed)
  torch.manual_seed(seed) # cpu 연산 무작위 고정
  torch.cuda.manual_seed(seed) # gpu 연산 무작위 고정
  torch.backends.cudnn.deterministic = True  # cuda 라이브러리에서 Deterministic(결정론적)으로 예측하기 (예측에 대한 불확실성 제거 )

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5630 entries, 0 to 5629
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   CustomerID                   5630 non-null   int64  
 1   Churn                        5630 non-null   int64  
 2   Tenure                       5366 non-null   float64
 3   PreferredLoginDevice         5630 non-null   object 
 4   CityTier                     5630 non-null   int64  
 5   WarehouseToHome              5379 non-null   float64
 6   PreferredPaymentMode         5630 non-null   object 
 7   Gender                       5630 non-null   object 
 8   HourSpendOnApp               5375 non-null   float64
 9   NumberOfDeviceRegistered     5630 non-null   int64  
 10  PreferedOrderCat             5630 non-null   object 
 11  SatisfactionScore            5630 non-null   int64  
 12  MaritalStatus                5630 non-null   object 
 13  NumberOfAddress   

In [5]:
df.isnull().sum().sort_values(ascending=False)

DaySinceLastOrder              307
OrderAmountHikeFromlastYear    265
Tenure                         264
OrderCount                     258
CouponUsed                     256
HourSpendOnApp                 255
WarehouseToHome                251
CustomerID                       0
PreferredLoginDevice             0
Churn                            0
PreferredPaymentMode             0
CityTier                         0
SatisfactionScore                0
PreferedOrderCat                 0
NumberOfDeviceRegistered         0
Gender                           0
Complain                         0
NumberOfAddress                  0
MaritalStatus                    0
CashbackAmount                   0
dtype: int64

In [6]:
# 중복 확인
print(f"중복 행 수: {df.duplicated().sum()}")

중복 행 수: 0


In [7]:
(df.isnull().sum() / df.shape[0]).round(4).sort_values(ascending=False) 

DaySinceLastOrder              0.0545
OrderAmountHikeFromlastYear    0.0471
Tenure                         0.0469
OrderCount                     0.0458
CouponUsed                     0.0455
HourSpendOnApp                 0.0453
WarehouseToHome                0.0446
CustomerID                     0.0000
PreferredLoginDevice           0.0000
Churn                          0.0000
PreferredPaymentMode           0.0000
CityTier                       0.0000
SatisfactionScore              0.0000
PreferedOrderCat               0.0000
NumberOfDeviceRegistered       0.0000
Gender                         0.0000
Complain                       0.0000
NumberOfAddress                0.0000
MaritalStatus                  0.0000
CashbackAmount                 0.0000
dtype: float64

In [8]:
null_list = df.columns[df.isnull().any()].tolist()
print(null_list)

['Tenure', 'WarehouseToHome', 'HourSpendOnApp', 'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder']


In [9]:
df[null_list].dtypes

Tenure                         float64
WarehouseToHome                float64
HourSpendOnApp                 float64
OrderAmountHikeFromlastYear    float64
CouponUsed                     float64
OrderCount                     float64
DaySinceLastOrder              float64
dtype: object

In [10]:
bins = [0, 10, 20, 30, 40]
labels = ['0~10', '11~20', '21~30', '31~40']

result = df[(df['Gender'] == 'Male') & (df['MaritalStatus'] == 'Single')][['Gender', 'MaritalStatus', 'Tenure']].copy()

result['Tenure_Group'] = pd.cut(result['Tenure'], bins=bins, labels=labels, include_lowest=True)

print("\n=== 그룹별 개수 (NaN 포함) ===")
print(result['Tenure_Group'].value_counts(dropna=False).sort_index())


=== 그룹별 개수 (NaN 포함) ===
Tenure_Group
0~10     641
11~20    246
21~30     93
31~40      5
NaN       53
Name: count, dtype: int64


In [11]:
import sys
sys.path.append('C:/dev/project/SKN27-2nd-4TEAM')

from src.missing_value.Missing_value_01 import fill_tenure, fill_day, fill_order_count, fill_hike, fill_coupon, fill_hour, fill_warehouse

# Tenure 결측치 제거
overall_median_ten = df['Tenure'].median()
df['Tenure'] = df.groupby('NumberOfAddress')['Tenure'].transform(
    lambda x: fill_tenure(x, overall_median_ten)
)
df['Tenure'].isnull().sum()  # 0이면 성공

np.int64(0)

In [12]:
overall_median_day = df['DaySinceLastOrder'].median()


df['DaySinceLastOrder'] = df.groupby(['PreferedOrderCat', 'PreferredLoginDevice'])['DaySinceLastOrder'].transform(lambda x: fill_day(x, overall_median_day))

df['DaySinceLastOrder'].isnull().sum()  # 0이면 성공

np.int64(0)

In [13]:
overall_median_ord = df['OrderCount'].median()


# 3. F값이 가장 높았던 컬럼으로 묶어서(groupby) 결측치 채우기
# (아래 리스트 안의 컬럼명은 1단계 결과에서 1, 2위를 한 컬럼명으로 바꿔주세요!)
best_cols = ['PreferedOrderCat', 'PreferredPaymentMode'] 

df['OrderCount'] = df.groupby(best_cols)['OrderCount'].transform(lambda x: fill_order_count(x, overall_median_ord))
# 람다 사용

# 4. 잘 채워졌는지 확인
df['OrderCount'].isnull().sum()  # 0이 나오면 성공!

np.int64(0)

In [14]:
# 1. 전체 중앙값 계산 (데이터가 너무 적은 그룹을 위한 방어용)
overall_median_hike = df['OrderAmountHikeFromlastYear'].median()

# 3. PreferedOrderCat 기준으로 그룹화하여 결측치 채우기
df['OrderAmountHikeFromlastYear'] = df.groupby('PreferedOrderCat')['OrderAmountHikeFromlastYear'].transform(lambda x: fill_hike(x, overall_median_hike))

# 4. 결과 확인
df['OrderAmountHikeFromlastYear'].isnull().sum()

np.int64(0)

In [15]:
(df.isnull().sum() / df.shape[0]).round(4).sort_values(ascending=False)

CouponUsed                     0.0455
HourSpendOnApp                 0.0453
WarehouseToHome                0.0446
CustomerID                     0.0000
PreferredLoginDevice           0.0000
Tenure                         0.0000
Churn                          0.0000
PreferredPaymentMode           0.0000
Gender                         0.0000
NumberOfDeviceRegistered       0.0000
PreferedOrderCat               0.0000
CityTier                       0.0000
SatisfactionScore              0.0000
MaritalStatus                  0.0000
Complain                       0.0000
NumberOfAddress                0.0000
OrderAmountHikeFromlastYear    0.0000
OrderCount                     0.0000
DaySinceLastOrder              0.0000
CashbackAmount                 0.0000
dtype: float64

In [16]:
# CouponUsed - PreferedOrderCat + PreferredLoginDevice 그룹
overall_median_coupon = df['CouponUsed'].median()

df['CouponUsed'] = df.groupby(['PreferedOrderCat', 'PreferredLoginDevice'])['CouponUsed'].transform(
    lambda x: fill_coupon(x, overall_median_coupon)
)

df['CouponUsed'].isnull().sum()

np.int64(0)

In [17]:
# HourSpendOnApp - PreferedOrderCat + PreferredLoginDevice 그룹
overall_median_hour = df['HourSpendOnApp'].median()

df['HourSpendOnApp'] = df.groupby(['PreferedOrderCat', 'PreferredLoginDevice'])['HourSpendOnApp'].transform(
    lambda x: fill_hour(x, overall_median_hour)
)

df['HourSpendOnApp'].isnull().sum()

np.int64(0)

In [18]:
# WarehouseToHome - PreferedOrderCat만 (2위 PreferredLoginDevice는 ❌)
overall_median_warehouse = df['WarehouseToHome'].median()

df['WarehouseToHome'] = df.groupby(['CityTier', 'PreferedOrderCat'])['WarehouseToHome'].transform(
    lambda x: fill_warehouse(x, overall_median_warehouse)
)

df['WarehouseToHome'].isnull().sum()

np.int64(0)

In [19]:
(df.isnull().sum() / df.shape[0]).round(4).sort_values(ascending=False)

CustomerID                     0.0
Churn                          0.0
Tenure                         0.0
PreferredLoginDevice           0.0
CityTier                       0.0
WarehouseToHome                0.0
PreferredPaymentMode           0.0
Gender                         0.0
HourSpendOnApp                 0.0
NumberOfDeviceRegistered       0.0
PreferedOrderCat               0.0
SatisfactionScore              0.0
MaritalStatus                  0.0
NumberOfAddress                0.0
Complain                       0.0
OrderAmountHikeFromlastYear    0.0
CouponUsed                     0.0
OrderCount                     0.0
DaySinceLastOrder              0.0
CashbackAmount                 0.0
dtype: float64

In [20]:
from src.outlier_values.outlier_control import outlier_control

cols = ['Tenure', 'WarehouseToHome', 'DaySinceLastOrder', 'CashbackAmount']

print("=== Before ===")
print(df[cols].describe())


=== Before ===
            Tenure  WarehouseToHome  DaySinceLastOrder  CashbackAmount
count  5630.000000      5630.000000        5630.000000     5630.000000
mean     10.051865        15.522380           4.662167      177.223030
std       8.407088         8.357004           3.600582       49.207036
min       0.000000         5.000000           0.000000        0.000000
25%       3.000000         9.000000           2.000000      145.770000
50%       8.000000        13.000000           4.000000      163.280000
75%      15.000000        20.000000           7.000000      196.392500
max      61.000000       127.000000          46.000000      324.990000


In [21]:
outlier_control(df, cols) # 이상치 제거


In [22]:
print(df.columns.tolist())  # 컬럼 확인

['CustomerID', 'Churn', 'PreferredLoginDevice', 'CityTier', 'PreferredPaymentMode', 'Gender', 'HourSpendOnApp', 'NumberOfDeviceRegistered', 'PreferedOrderCat', 'SatisfactionScore', 'MaritalStatus', 'NumberOfAddress', 'Complain', 'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount', 'Tenure_log', 'WarehouseToHome_log', 'DaySinceLastOrder_clip', 'CashbackAmount_clip']


In [23]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5630 entries, 0 to 5629
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   CustomerID                   5630 non-null   int64  
 1   Churn                        5630 non-null   int64  
 2   PreferredLoginDevice         5630 non-null   object 
 3   CityTier                     5630 non-null   int64  
 4   PreferredPaymentMode         5630 non-null   object 
 5   Gender                       5630 non-null   object 
 6   HourSpendOnApp               5630 non-null   float64
 7   NumberOfDeviceRegistered     5630 non-null   int64  
 8   PreferedOrderCat             5630 non-null   object 
 9   SatisfactionScore            5630 non-null   int64  
 10  MaritalStatus                5630 non-null   object 
 11  NumberOfAddress              5630 non-null   int64  
 12  Complain                     5630 non-null   int64  
 13  OrderAmountHikeFro